In [ ]:
from expert_data_generation.dense_trajectories import (
    CircleObstacle, generate_grid_set, generate_dense_set, export_csv
)
from expert_data_generation.model import RobotModel, DiffDriveKinematics
import numpy as np
import matplotlib.pyplot as plt

In [ ]:

robot = RobotModel(wheel_radius=0.05, wheel_base=0.3)
kin = DiffDriveKinematics(robot)

obstacle = CircleObstacle(2.0, 2.0, 0.5)
start = np.array([0.0, 0.0, 0.0])
goal  = np.array([4.0, 4.0, np.pi / 2])

print("Generating streamline grid (fills space) ...")
trajs_grid = generate_grid_set(
    kin, start, goal, obstacle,
    grid_nx=12, grid_ny=12,
    bbox_margin=0.6, grid_min_clearance=0.05,
    clearance=0.005, w_effort=1.0, w_smooth=20.0,
    N_steps=160, dt=0.1,
    v_bounds=(-0.2, 0.5), omega_bounds=(-2.0, 2.0),
    dedup_thresh=0.05, max_length_factor=2.5,
)
print("Generating wall-hugging polar set (follows curvature) ...")
trajs_hug = generate_dense_set(
    kin, start, goal, obstacle,
    n_angles=36,                            # 10° resolution around obstacle
    radius_offsets=(0.04, 0.12, 0.25),      # tightest still > clearance (0.005) → no boundary wobble
    clearances=(0.005,),                    # near-zero margin from the circle
    cost_grid=((1.0, 20.0),),               # smooth-dominant → no kinks
    hug_strengths=(8.0,),                   # strong pull onto the wall
    arc_spans_deg=(20.0, 40.0),             # distributed pull, both well under wrap
    N_steps=160, dt=0.1,
    v_bounds=(-0.2, 0.5), omega_bounds=(-2.0, 2.0),
    dedup_thresh=0.05, max_length_factor=2.5,
)
trajs = trajs_grid + trajs_hug
print(f"Grid: {len(trajs_grid)}  Hugging: {len(trajs_hug)}  Total: {len(trajs)}")
print(f"\n{len(trajs)} unique trajectories generated.")

out_csv = "/home/bb/Desktop/atic-cbfs/results/dense_trajectories.csv"
n_rows = export_csv(trajs, out_csv)
print(f"Wrote {n_rows} (state, action) rows to {out_csv}")

# Plot
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.add_patch(plt.Circle((obstacle.x, obstacle.y), obstacle.radius,
                            color="tab:red", alpha=0.5, zorder=5))
    # Show all trajectories with alpha for density visualization
    for t in trajs:
        ax.plot(t.states[:, 0], t.states[:, 1], "-",
                color="tab:blue", alpha=0.25, linewidth=0.9)
    ax.plot(*start[:2], "go", markersize=14, label="start", zorder=10)
    ax.plot(*goal[:2],  "r*", markersize=18, label="goal", zorder=10)
    ax.set_aspect("equal")
    ax.set_title(f"{len(trajs)} dense expert trajectories around single obstacle")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig("/home/bb/Desktop/atic-cbfs/results/dense_trajectories.png", dpi=110)
    print("Saved /results/dense_trajectories.png")
except Exception as e:
    print(f"(plot skipped: {e})")